# Full pipeline

Same behavior as `main.py` / root `main.ipynb`.

- `FOLDER`: one issue folder name under `Input/`, or `None` for all.
- `RUN_EVAL`: if True, after converter run `evaluation.py` (Output vs `Input_Eval`).


In [ ]:
from pathlib import Path
import os
import sys

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "Input").is_dir():
        return cwd
    if (cwd.parent / "Input").is_dir():
        return cwd.parent
    return cwd

PROJECT_ROOT = resolve_project_root()
repo_root_str = str(PROJECT_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)
os.chdir(PROJECT_ROOT)

FOLDER = None  # e.g. 'ajil0120no1'
RUN_EVAL = True


In [ ]:
import logging
import os
import sys
from pathlib import Path
from typing import Optional

from Combinator.combine_results_v2 import combine_folder as combine_folder_results
from Converter.yml_manager import process_journal as convert_journal_result
from CrossRef.extract import process_folder
from Utilities.app_logging import setup_logging
from Webscraper.databaseScrape import process as process_database_fallback
from Webscraper.recursiveScrape import scrape_from_database_url
from Webscraper.scraper import scrape_from_crossref_result, scrape_without_crossref
from LLM.ocr_pipeline import run_ocr_pipeline
from evaluation import evaluationScore


In [ ]:
def wait_for_ocr_pipeline(folder_name, ocr_process):
    logging.info(
        'Waiting for OCR pipeline process to finish for folder=%s child_pid=%s',
        folder_name,
        ocr_process.pid,
    )
    stdout, stderr = ocr_process.communicate()
    if ocr_process.returncode == 0:
        logging.info('OCR pipeline completed for folder=%s pid=%s', folder_name, ocr_process.pid)
    else:
        logging.error(
            'OCR pipeline failed for folder=%s pid=%s returncode=%s stderr=%s',
            folder_name, ocr_process.pid, ocr_process.returncode, stderr.strip(),
        )
    if stdout.strip():
        logging.info('OCR pipeline stdout for folder=%s:\n%s', folder_name, stdout.strip())
    if stderr.strip() and ocr_process.returncode == 0:
        logging.warning('OCR pipeline stderr for folder=%s:\n%s', folder_name, stderr.strip())


In [ ]:
def run_pipeline(project_root: Path, folder: Optional[str] = None, run_eval: bool = True) -> None:
    log_file = setup_logging()
    input_dir = project_root / 'Input'
    crossref_results_dir = project_root / 'CrossRef' / 'results'
    webscraper_output_dir = project_root / 'Webscraper' / 'output'
    webscraper_results_dir = project_root / 'Webscraper' / 'results'
    process_id = os.getpid()
    logging.info('Application started with pid=%s', process_id)
    print(f'Logging to {log_file}')
    if not input_dir.exists():
        logging.error('Input folder not found: %s', input_dir)
        print(f'Input folder not found: {input_dir}')
        return
    processed_count = 0
    folders = sorted(path for path in input_dir.iterdir() if path.is_dir())
    if folder:
        target_folder = input_dir / folder
        if not target_folder.exists() or not target_folder.is_dir():
            logging.error('Requested folder not found in Input: %s', target_folder)
            print(f'Requested folder not found in Input: {target_folder}')
            return
        folders = [target_folder]
    for f in folders:
        logging.info('Dispatching folder=%s main_pid=%s', f.name, process_id)
        process_folder(f.name, str(f))
        crossref_result_file = crossref_results_dir / f'{f.name}.json'
        if crossref_result_file.exists():
            logging.info('Starting Webscraper for folder=%s using CrossRef result=%s', f.name, crossref_result_file)
            scrape_from_crossref_result(f.name, str(crossref_result_file), output_dir=str(webscraper_output_dir), results_dir=str(webscraper_results_dir))
            processed_count += 1
        else:
            logging.info('No CrossRef result found for folder=%s; starting databaseScrape fallback', f.name)
            fallback_result = process_database_fallback(f.name)
            logging.info('databaseScrape fallback completed for folder=%s status=%s url=%s', f.name, fallback_result.get('status'), fallback_result.get('url'))
            if fallback_result.get('status') == 'matched':
                logging.info('Starting recursiveScrape for folder=%s using databaseScrape url=%s', f.name, fallback_result.get('url'))
                scrape_from_database_url(f.name, fallback_result['url'], output_dir=str(webscraper_output_dir), results_dir=str(webscraper_results_dir))
                processed_count += 1
            else:
                scrape_without_crossref(f.name)
        print(f'Input Directory: {input_dir}, Folder: {f.name}')
        ocr_input_dir = os.path.join(input_dir, f.name)
        print(f'Running OCR pipeline for folder: {ocr_input_dir}')
        run_ocr_pipeline(ocr_input_dir)
        logging.info('Starting combinator for folder=%s after OCR wait', f.name)
        combine_folder_results(f.name)
        logging.info('Finished combinator for folder=%s', f.name)
        logging.info('Starting converter for folder=%s after combinator', f.name)
        convert_output_path = convert_journal_result(f.name)
        if convert_output_path:
            logging.info('Finished converter for folder=%s output=%s', f.name, convert_output_path)
            if run_eval:
                eval_gt = project_root / 'Input_Eval' / f.name / 'structure.yml'
                if eval_gt.exists():
                    try:
                        logging.info('Starting evaluation for folder=%s', f.name)
                        evaluationScore(str(Path(convert_output_path).resolve()))
                    except Exception:
                        logging.exception('Evaluation failed for folder=%s', f.name)
                else:
                    logging.warning('Skipping evaluation: ground truth not found at %s', eval_gt)
        else:
            logging.warning('Converter did not generate output for folder=%s', f.name)
    logging.info('Finished processing %s folder(s)', processed_count)
    print(f'Finished processing {processed_count} folder(s)')


In [ ]:
run_pipeline(PROJECT_ROOT, folder=FOLDER, run_eval=RUN_EVAL)
